# 23特徴量の品質点検と低段位期・高段位期の記述的比較

しょぼんｎの段位16以下=0、18以上=1を維持。段位17は採点外・履歴には保持。
既存H1〜H6、元CSV、モデルは変更しません。ZIP解析・モデル学習・予測は行いません。
基準コードは `c6e149986a3ac63ccc12a22dff03e0c4084bf7c5`。使うのはしょぼんnのCSVとH3/H4の監査成果物だけです。
外部評価スコアは読み込まず、共通対局IDはH4と同じ対象を再現するためだけに使用します。

**実行順：A → B（設定）→ C → D → E → F。H1〜H6の再実行は不要。**
Bに既存の `validation_reports/hourly_*` フォルダを指定します（model_*やdiagnostics_*ではありません）。
H4の窓一覧がなければ、主方式との一致を確認できないため停止します。
主結果はhour_invariantの全学習対象の記述統計であり、交差検証の性能評価ではありません。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import ast, json, re, uuid, hashlib
from collections import Counter
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT=Path('/content/drive/MyDrive/卒論')
H_REPORT_DIR=None  # 必須：例 ROOT/'validation_reports/hourly_...'
SHOBON_CACHE_DIR=None  # 通常None。元のキャッシュが移動した場合のみ指定
WINDOW=10
INITIAL_EXCLUDE=1500

In [ ]:
BASE_FEATURES = [
    'first_discard_raw_unsafe_rate', 'first_discard_safe_rate',
    'early_noten_push_rate', 'mid_noten_push_rate', 'late_noten_push_rate',
    'las_desperate_push_rate', 'top_conservative_push_rate', 'flat_situation_push_rate',
    'riichi_dora_mean', 'chasing_riichi_good_shape_rate', 'chasing_riichi_low_value_rate',
    'chasing_riichi_rate', 'early_meld_ratio', 'late_meld_ratio', 'meld_dora_ratio',
    'raw_unsafe_discard_rate', 'ryuukyoku_tenpai_rate', 'ryuukyoku_noten_safe_rate',
]

CROSS_FEATURES = ['top_late_noten_push_rate', 'las_early_meld_rate',
                  'early_meld_with_dora_rate', 'chasing_riichi_quality_diff',
                  'flat_first_discard_safe_rate']

FEATURES = BASE_FEATURES + CROSS_FEATURES

class ValidationBlocked(ValueError):
    pass

def require(condition, message):
    if not condition:
        raise ValidationBlocked(message)

def canonical_game_id(value):
    # パス・視点tw・拡張子を除去。同じ対局の別視点も同じIDにする。
    name = str(value).replace('\\', '/').rsplit('/', 1)[-1].lower()
    match = re.fullmatch(r'(\d{10}gm-[0-9a-f]+-[0-9a-f]+-[0-9a-f]+)(?:&tw=[0-3])?(?:\.mjlog)?', name)
    if not match:
        raise ValidationBlocked(f'解釈できない対局ID: {value}')
    return match.group(1)

def fit_imputation(training_rows):
    # 明示された学習区間だけ。外部データや検証区間は渡さない。
    require(len(training_rows) > 0, '補完平均の学習対象が空です')
    means = training_rows[BASE_FEATURES].mean()
    require(np.isfinite(means.to_numpy()).all(),
            '学習区間で全欠損の特徴量があります: ' + str(means.index[means.isna()].tolist()))
    return means

def add_cross(base):
    # 元ノートブックの5式をそのまま維持。局面別再集計はしない。
    x = base[BASE_FEATURES].copy()
    x['top_late_noten_push_rate'] = x['top_conservative_push_rate'] * x['late_noten_push_rate']
    x['las_early_meld_rate'] = x['las_desperate_push_rate'] * x['early_meld_ratio']
    x['early_meld_with_dora_rate'] = x['early_meld_ratio'] * x['meld_dora_ratio']
    x['chasing_riichi_quality_diff'] = x['chasing_riichi_good_shape_rate'] - x['chasing_riichi_low_value_rate']
    x['flat_first_discard_safe_rate'] = x['first_discard_safe_rate'] * (1.0 - x['flat_situation_push_rate'])
    return x[FEATURES]

def arrange_hours(data,scenario):
    result=data.copy()
    if scenario=='hash_seed42':
        result['_tie']=result.game_id.map(lambda x:hashlib.sha256(('42:'+x).encode()).hexdigest())
    else:
        result['_tie']=result.game_id
    result=result.sort_values(['hour','_tie'],ascending=[True,scenario!='id_descending']).drop(columns='_tie').reset_index(drop=True)
    return result

def hour_means(rows,cutoff,forbidden):
    fitting=rows[(rows.hour>cutoff)&~rows.game_id.isin(forbidden)]
    means=fit_imputation(fitting)
    return means,set(fitting.game_id)

def prepare_hour_block(rows,means,cutoff,scenario,forbidden=frozenset()):
    require(len(rows)>=WINDOW,'区間に10対局未満しかありません')
    require(rows.game_id.is_unique,'区間の対局ID重複')
    ordered=arrange_hours(rows,scenario)
    hour_sizes=ordered.groupby('hour').size()
    filled=ordered[BASE_FEATURES].fillna(means)
    require(np.isfinite(filled.to_numpy()).all(),'補完後に非有限値')
    rolled=add_cross(filled).rolling(WINDOW,min_periods=WINDOW).mean()
    ids=ordered.game_id.tolist();hours=ordered.hour.tolist()
    metadata=[];keep=[];excluded=[]
    for i,row in ordered.iterrows():
        start=i-WINDOW+1
        source=tuple(ids[max(0,start):i+1])
        reason=''
        if start<0:reason='区間先頭9対局・履歴専用'
        elif row.hour<=cutoff:reason='観測初期1500対局と同時間帯・採点外'
        elif pd.isna(row.target_label):reason='段位17・履歴のみ'
        elif set(source)&set(forbidden):reason='外部共通対局を含む学習窓'
        elif scenario=='hour_invariant':
            if hour_sizes.loc[row.hour]!=1:reason='当該対局の時間帯内順序が未確定'
            elif start>0 and hours[start-1]==hours[start]:reason='窓の開始位置が時間帯の途中'
        if reason:
            excluded.append({'game_id':row.game_id,'reason':reason})
        else:
            keep.append(i)
            metadata.append({'game_id':row.game_id,'hour':row.hour,'dan':row.dan,
                             'target_label':int(row.target_label),'source_ids':source,
                             'source_hours':tuple(str(x) for x in hours[start:i+1])})
    meta=pd.DataFrame(metadata,columns=['game_id','hour','dan','target_label','source_ids','source_hours'])
    return dict(X=rolled.iloc[keep].reset_index(drop=True),y=meta.target_label.astype(int),meta=meta,
                excluded=pd.DataFrame(excluded,columns=['game_id','reason']))

CATALOG=pd.DataFrame([{'feature': 'first_discard_raw_unsafe_rate', '意味': '他家リーチ後の初打非現物率', '分子または左項': '初打で最後のリーチ者の河にない牌を切った回数', '分母または右項': '他家REACH step=1の回数', '計算式': '(初打で最後のリーチ者の河にない牌を切った回数) / (他家REACH step=1の回数)', '解釈': '他家リーチ後に非現物を選ぶ傾向', '定義上の懸念': 'スジ判定はpassで未実装。複数リーチでは最後の宣言者だけを見る。打牌前に局終了すると分母だけ増える', '提案': '無筋という名称との対応、スジ・複数リーチ・初打機会の分母を点検', '優先度': 1, '区分': '修正検討', '欠損条件': '分母0ならNaN（元CSVの事象分母の数自体は未保存）'}, {'feature': 'first_discard_safe_rate', '意味': '他家リーチ後の初打現物率', '分子または左項': '初打で最後のリーチ者の現物を切った回数', '分母または右項': '他家REACH step=1の回数', '計算式': '(初打で最後のリーチ者の現物を切った回数) / (他家REACH step=1の回数)', '解釈': '直後に現物を選ぶ傾向', '定義上の懸念': '現物打牌はベタオリの意思そのものではない。分母は実打牌機会数ではない', '提案': '複数リーチと局終了を含めた機会定義を点検', '優先度': 2, '区分': '修正検討', '欠損条件': '分母0ならNaN（元CSVの事象分母の数自体は未保存）'}, {'feature': 'early_noten_push_rate', '意味': '序盤の門前未リーチ時非現物率', '分子または左項': '自分のツモ回数0〜6・他家リーチ中・門前未リーチで非現物を打った回数', '分母または右項': '自分のツモ回数0〜6・他家リーチ中・門前未リーチの自分の打牌数', '計算式': '(自分のツモ回数0〜6・他家リーチ中・門前未リーチで非現物を打った回数) / (自分のツモ回数0〜6・他家リーチ中・門前未リーチの自分の打牌数)', '解釈': '門前未リーチ状態での押しの代理指標', '定義上の懸念': 'ノーテンはis_menzen and not my_reach_processedで代用。ダマテンを含み副露ノーテンを除く。巡目はツモ回数', '提案': '実シャンテン判定と副露状態・巡目の扱いを検証', '優先度': 1, '区分': '修正検討', '欠損条件': '分母0ならNaN（元CSVの事象分母の数自体は未保存）'}, {'feature': 'mid_noten_push_rate', '意味': '中盤の門前未リーチ時非現物率', '分子または左項': '自分のツモ回数7〜12・他家リーチ中・門前未リーチで非現物を打った回数', '分母または右項': '自分のツモ回数7〜12・他家リーチ中・門前未リーチの自分の打牌数', '計算式': '(自分のツモ回数7〜12・他家リーチ中・門前未リーチで非現物を打った回数) / (自分のツモ回数7〜12・他家リーチ中・門前未リーチの自分の打牌数)', '解釈': '門前未リーチ状態での押しの代理指標', '定義上の懸念': 'ノーテンはis_menzen and not my_reach_processedで代用。ダマテンを含み副露ノーテンを除く。巡目はツモ回数', '提案': '実シャンテン判定と副露状態・巡目の扱いを検証', '優先度': 1, '区分': '修正検討', '欠損条件': '分母0ならNaN（元CSVの事象分母の数自体は未保存）'}, {'feature': 'late_noten_push_rate', '意味': '終盤の門前未リーチ時非現物率', '分子または左項': '自分のツモ回数13以上・他家リーチ中・門前未リーチで非現物を打った回数', '分母または右項': '自分のツモ回数13以上・他家リーチ中・門前未リーチの自分の打牌数', '計算式': '(自分のツモ回数13以上・他家リーチ中・門前未リーチで非現物を打った回数) / (自分のツモ回数13以上・他家リーチ中・門前未リーチの自分の打牌数)', '解釈': '門前未リーチ状態での押しの代理指標', '定義上の懸念': 'ノーテンはis_menzen and not my_reach_processedで代用。ダマテンを含み副露ノーテンを除く。巡目はツモ回数', '提案': '実シャンテン判定と副露状態・巡目の扱いを検証', '優先度': 1, '区分': '修正検討', '欠損条件': '分母0ならNaN（元CSVの事象分母の数自体は未保存）'}, {'feature': 'las_desperate_push_rate', '意味': 'ラス目での非現物率', '分子または左項': '局開始時4位で他家リーチ中の非現物打牌数', '分母または右項': '局開始時4位で他家リーチ中の自分の打牌数', '計算式': '(局開始時4位で他家リーチ中の非現物打牌数) / (局開始時4位で他家リーチ中の自分の打牌数)', '解釈': '点数順位別の押し方の代理指標', '定義上の懸念': '点数はINIT時のみ。同点は自分より高い点数の人数で順位計算、全員同点なら全員1位。平場は点差を見ない', '提案': '同点・点差・局面中の点数変動と複数リーチへの安全判定を点検', '優先度': 2, '区分': '修正検討', '欠損条件': '分母0ならNaN（元CSVの事象分母の数自体は未保存）'}, {'feature': 'top_conservative_push_rate', '意味': 'トップ目での非現物率', '分子または左項': '局開始時1位で他家リーチ中の非現物打牌数', '分母または右項': '局開始時1位で他家リーチ中の自分の打牌数', '計算式': '(局開始時1位で他家リーチ中の非現物打牌数) / (局開始時1位で他家リーチ中の自分の打牌数)', '解釈': '点数順位別の押し方の代理指標', '定義上の懸念': '点数はINIT時のみ。同点は自分より高い点数の人数で順位計算、全員同点なら全員1位。平場は点差を見ない', '提案': '同点・点差・局面中の点数変動と複数リーチへの安全判定を点検', '優先度': 2, '区分': '修正検討', '欠損条件': '分母0ならNaN（元CSVの事象分母の数自体は未保存）'}, {'feature': 'flat_situation_push_rate', '意味': '中位での非現物率', '分子または左項': '局開始時2・3位で他家リーチ中の非現物打牌数', '分母または右項': '局開始時2・3位で他家リーチ中の自分の打牌数', '計算式': '(局開始時2・3位で他家リーチ中の非現物打牌数) / (局開始時2・3位で他家リーチ中の自分の打牌数)', '解釈': '点数順位別の押し方の代理指標', '定義上の懸念': '点数はINIT時のみ。同点は自分より高い点数の人数で順位計算、全員同点なら全員1位。平場は点差を見ない', '提案': '同点・点差・局面中の点数変動と複数リーチへの安全判定を点検', '優先度': 2, '区分': '修正検討', '欠損条件': '分母0ならNaN（元CSVの事象分母の数自体は未保存）'}, {'feature': 'riichi_dora_mean', '意味': '自分のリーチ時の簡易ドラ平均枚数', '分子または左項': '自分の最初のREACH時の手牌中の表ドラ枚数の合計', '分母または右項': '自分のリーチ回数', '計算式': '(自分の最初のREACH時の手牌中の表ドラ枚数の合計) / (自分のリーチ回数)', '解釈': 'リーチ時に保持する表ドラ量の代理指標', '定義上の懸念': '宣言打牌前の手牌を数え、切るドラを含み得る。赤ドラ・重複表示牌の倍数は反映しない', '提案': '宣言打牌後の手牌、赤ドラ・重複ドラを点検', '優先度': 1, '区分': '修正検討', '欠損条件': '分母0ならNaN（元CSVの事象分母の数自体は未保存）'}, {'feature': 'chasing_riichi_good_shape_rate', '意味': '追いかけリーチ時の簡易良形率', '分子または左項': '追いかけ時に手牌牌種数が10以上だった回数', '分母または右項': '追いかけリーチ回数', '計算式': '(追いかけ時に手牌牌種数が10以上だった回数) / (追いかけリーチ回数)', '解釈': '手牌の牌種多様性を用いた待ち形の代理指標', '定義上の懸念': '両面待ちや待ち牌数は計算していない。14枚時点の牌種数による近似', '提案': '実際の待ち形・待ち牌数との対応を確認', '優先度': 1, '区分': '修正検討', '欠損条件': '分母0ならNaN（元CSVの事象分母の数自体は未保存）'}, {'feature': 'chasing_riichi_low_value_rate', '意味': '追いかけリーチ時のドラ0率', '分子または左項': '追いかけ時の簡易ドラ枚数が0の回数', '分母または右項': '追いかけリーチ回数', '計算式': '(追いかけ時の簡易ドラ枚数が0の回数) / (追いかけリーチ回数)', '解釈': '表ドラなしで追いかける傾向', '定義上の懸念': '役・翻数・赤ドラは見ないので安手とは限らない', '提案': 'ドラ0と低打点を区別して名称・手価値判定を検証', '優先度': 1, '区分': '修正検討', '欠損条件': '分母0ならNaN（元CSVの事象分母の数自体は未保存）'}, {'feature': 'chasing_riichi_rate', '意味': '他家リーチ宣言数に対する追いかけ回数比', '分子または左項': '他家リーチが既にある状態での自分のリーチ回数', '分母または右項': '他家REACH step=1の回数', '計算式': '(他家リーチが既にある状態での自分のリーチ回数) / (他家REACH step=1の回数)', '解釈': '追いかけの頻度比', '定義上の懸念': '分母は他家リーチ局数でも自分の全リーチ数でもない。複数宣言で増える', '提案': '現行の回数比として記述し、機会分母を将来点検', '優先度': 2, '区分': '維持', '欠損条件': '分母0ならNaN（元CSVの事象分母の数自体は未保存）'}, {'feature': 'early_meld_ratio', '意味': '序盤の初回N発生局率', '分子または左項': '初回Nが自分のツモ回数6以下の局数', '分母または右項': '全INIT数', '計算式': '(初回Nが自分のツモ回数6以下の局数) / (全INIT数)', '解釈': '初回仕掛けの時期またはドラ保持を表す代理指標', '定義上の懸念': 'Nの復号がなく暗槓等も含む。副露後の手牌更新なし。ドラ率の分母は副露局数ではなく全局', '提案': '副露種類・手牌更新・鳴き巡目・ドラの対象と分母を点検', '優先度': 1, '区分': '修正検討', '欠損条件': '全INIT数0なら0（非発生と区別できない）'}, {'feature': 'late_meld_ratio', '意味': '終盤の初回N発生局率', '分子または左項': '初回Nが自分のツモ回数13以上の局数', '分母または右項': '全INIT数', '計算式': '(初回Nが自分のツモ回数13以上の局数) / (全INIT数)', '解釈': '初回仕掛けの時期またはドラ保持を表す代理指標', '定義上の懸念': 'Nの復号がなく暗槓等も含む。副露後の手牌更新なし。ドラ率の分母は副露局数ではなく全局', '提案': '副露種類・手牌更新・鳴き巡目・ドラの対象と分母を点検', '優先度': 1, '区分': '修正検討', '欠損条件': '全INIT数0なら0（非発生と区別できない）'}, {'feature': 'meld_dora_ratio', '意味': '初回N時に表ドラを持つ局率', '分子または左項': '初回N時の自分の保持牌に表ドラがある局数', '分母または右項': '全INIT数', '計算式': '(初回N時の自分の保持牌に表ドラがある局数) / (全INIT数)', '解釈': '初回仕掛けの時期またはドラ保持を表す代理指標', '定義上の懸念': 'Nの復号がなく暗槓等も含む。副露後の手牌更新なし。ドラ率の分母は副露局数ではなく全局', '提案': '副露種類・手牌更新・鳴き巡目・ドラの対象と分母を点検', '優先度': 1, '区分': '修正検討', '欠損条件': '全INIT数0なら0（非発生と区別できない）'}, {'feature': 'raw_unsafe_discard_rate', '意味': '他家リーチ中の全体非現物率', '分子または左項': '他家リーチ中の非現物打牌数', '分母または右項': '他家リーチ中の自分の全打牌数', '計算式': '(他家リーチ中の非現物打牌数) / (他家リーチ中の自分の全打牌数)', '解釈': '他家リーチに対する全体の押しの代理指標', '定義上の懸念': '無筋・スジを分けず、最後のリーチ者の河だけを見る', '提案': '非現物という実定義に沿って解釈し、安全判定を点検', '優先度': 1, '区分': '修正検討', '欠損条件': '分母0ならNaN（元CSVの事象分母の数自体は未保存）'}, {'feature': 'ryuukyoku_tenpai_rate', '意味': '流局時の手牌開示率', '分子または左項': 'RYUUKYOKUの本人hai属性が非空の回数', '分母または右項': '全RYUUKYOKU数', '計算式': '(RYUUKYOKUの本人hai属性が非空の回数) / (全RYUUKYOKU数)', '解釈': '流局時テンパイの代理指標', '定義上の懸念': '流局種類を区別せず、開示属性をテンパイ扱い。安全に粘ったかは未計測', '提案': '流局種類と開示属性を確認', '優先度': 2, '区分': '維持', '欠損条件': '分母0ならNaN（元CSVの事象分母の数自体は未保存）'}, {'feature': 'ryuukyoku_noten_safe_rate', '意味': '流局時の手牌非開示率', '分子または左項': 'RYUUKYOKUの本人hai属性が空の回数', '分母または右項': '全RYUUKYOKU数', '計算式': '(RYUUKYOKUの本人hai属性が空の回数) / (全RYUUKYOKU数)', '解釈': '手牌開示率の補数', '定義上の懸念': 'テンパイ率と和が1。流局した局だけを母集団にし、放銃回避成功率は測っていない', '提案': '安全率との名称対応と独立した守備指標の必要性を検討', '優先度': 1, '区分': '修正検討', '欠損条件': '分母0ならNaN（元CSVの事象分母の数自体は未保存）'}, {'feature': 'top_late_noten_push_rate', '意味': 'トップ非現物率×終盤門前未リーチ非現物率', '分子または左項': 'top_conservative_push_rate', '分母または右項': 'late_noten_push_rate', '計算式': 'top_conservative_push_rate * late_noten_push_rate', '解釈': '試合ごとの既存比率の数値的な交互作用', '定義上の懸念': '同一局面の条件付き割合ではない。構成特徴量の近似・欠損補完の影響を引き継ぐ', '提案': '数値的な積・差として記述し、同一局面の意味を与えない。構成要素の品質を優先', '優先度': 2, '区分': '修正検討', '欠損条件': '補完前の診断では片方欠損ならNaN。モデル処理は基本18を補完した後に計算'}, {'feature': 'las_early_meld_rate', '意味': 'ラス非現物率×序盤初回N局率', '分子または左項': 'las_desperate_push_rate', '分母または右項': 'early_meld_ratio', '計算式': 'las_desperate_push_rate * early_meld_ratio', '解釈': '試合ごとの既存比率の数値的な交互作用', '定義上の懸念': '同一局面の条件付き割合ではない。構成特徴量の近似・欠損補完の影響を引き継ぐ', '提案': '数値的な積・差として記述し、同一局面の意味を与えない。構成要素の品質を優先', '優先度': 2, '区分': '修正検討', '欠損条件': '補完前の診断では片方欠損ならNaN。モデル処理は基本18を補完した後に計算'}, {'feature': 'early_meld_with_dora_rate', '意味': '序盤初回N局率×初回N時ドラ保持局率', '分子または左項': 'early_meld_ratio', '分母または右項': 'meld_dora_ratio', '計算式': 'early_meld_ratio * meld_dora_ratio', '解釈': '試合ごとの既存比率の数値的な交互作用', '定義上の懸念': '同一局面の条件付き割合ではない。構成特徴量の近似・欠損補完の影響を引き継ぐ', '提案': '数値的な積・差として記述し、同一局面の意味を与えない。構成要素の品質を優先', '優先度': 2, '区分': '修正検討', '欠損条件': '補完前の診断では片方欠損ならNaN。モデル処理は基本18を補完した後に計算'}, {'feature': 'chasing_riichi_quality_diff', '意味': '追いかけ簡易良形率−ドラ0率', '分子または左項': 'chasing_riichi_good_shape_rate', '分母または右項': 'chasing_riichi_low_value_rate', '計算式': 'chasing_riichi_good_shape_rate - chasing_riichi_low_value_rate', '解釈': '試合ごとの既存比率の数値的な交互作用', '定義上の懸念': '同一局面の条件付き割合ではない。構成特徴量の近似・欠損補完の影響を引き継ぐ', '提案': '数値的な積・差として記述し、同一局面の意味を与えない。構成要素の品質を優先', '優先度': 2, '区分': '修正検討', '欠損条件': '補完前の診断では片方欠損ならNaN。モデル処理は基本18を補完した後に計算'}, {'feature': 'flat_first_discard_safe_rate', '意味': '初打現物率×(1−中位非現物率)', '分子または左項': 'first_discard_safe_rate', '分母または右項': '(1-flat_situation_push_rate)', '計算式': 'first_discard_safe_rate * (1-flat_situation_push_rate)', '解釈': '試合ごとの既存比率の数値的な交互作用', '定義上の懸念': '同一局面の条件付き割合ではない。構成特徴量の近似・欠損補完の影響を引き継ぐ', '提案': '数値的な積・差として記述し、同一局面の意味を与えない。構成要素の品質を優先', '優先度': 2, '区分': '修正検討', '欠損条件': '補完前の診断では片方欠損ならNaN。モデル処理は基本18を補完した後に計算'}])

## D：保存CSVの品質点検・主方式の再現

欠損は0と区別し、元データを補完・改変しません。事象分母の件数は元CSVに保存されていないため復元しません。
品質点検は基本18と、補完前に既存5式を適用した交差項を対象にします。
主方式は既存H4と同じ「基本18の平均補完→交差5→10対局平均」を使います。
補完値はしょぼんnの観測初期期間後・共通対局以外だけから算出。ラベル別の補完はしません。

In [ ]:
require(H_REPORT_DIR is not None,'BでH_REPORT_DIRを指定してください')
report=Path(H_REPORT_DIR)
facts_path=report/'history_and_cache_audit.csv'
common_path=report/'common_games_EXCLUDE_FROM_TRAINING_SOURCES.csv'
windows_path=report/'hour_invariant_aligned_final_train_windows.csv'
if not windows_path.is_file():windows_path=report/'hour_invariant_final_train_windows.csv'
for p in [facts_path,common_path,windows_path]:require(p.is_file(),f'必要なH3/H4成果物がありません: {p}')
facts=pd.read_csv(facts_path)
sfacts=facts[facts.player.eq('shobon')]
require(len(sfacts)==1,'しょぼんnの入力出所が一意ではありません')
cache=Path(SHOBON_CACHE_DIR) if SHOBON_CACHE_DIR is not None else Path(sfacts.iloc[0].cache_dir)
paths=[cache/'raw_manifest.csv',cache/'raw_features.csv']
for p in paths:require(p.is_file(),f'必要な保存CSVがありません: {p}')
def digest(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
require(digest(paths[0])==sfacts.iloc[0].raw_manifest_sha256 and digest(paths[1])==sfacts.iloc[0].raw_features_sha256,
        'CSVがH3実行時と一致しません。別データを混在させず確認してください')
out=ROOT/'feature_analysis'/('shobon_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:8])
out.mkdir(parents=True,exist_ok=False)
CATALOG.to_csv(out/'feature_definitions.csv',index=False,encoding='utf-8-sig')
m=pd.read_csv(paths[0]);f=pd.read_csv(paths[1])
require({'file_id','dan'}<=set(m.columns) and {'file_id',*BASE_FEATURES}<=set(f.columns),'入力列不足')
for frame in [m,f]:
    frame['game_id']=frame.file_id.map(canonical_game_id)
    require(frame.game_id.is_unique,'正規化対局IDが重複しています')
require(set(m.game_id)==set(f.game_id),'台帳と特徴量に不足対局があります')
raw_numeric=f[BASE_FEATURES].apply(pd.to_numeric,errors='coerce')
bad_text=f[BASE_FEATURES].notna() & raw_numeric.isna()
f[BASE_FEATURES]=raw_numeric
d=m.drop(columns=BASE_FEATURES,errors='ignore').merge(f[['game_id']+BASE_FEATURES],on='game_id',validate='one_to_one')
d['hour']=pd.to_datetime(d.game_id.str[:10],format='%Y%m%d%H',errors='coerce')
d['dan']=pd.to_numeric(d.dan,errors='coerce')
require(d.hour.notna().all() and d.dan.between(0,20).all() and d.dan.mod(1).eq(0).all(),'日時・段位が不正')
d=d.sort_values(['hour','game_id']).reset_index(drop=True)
require(len(d)>1500,'初期期間後の対局不足')
cutoff=d.iloc[1499].hour
require(str(cutoff)==str(pd.Timestamp(sfacts.iloc[0].cutoff_hour)),'初期期間の境界がH3と不一致')
d['target_label']=np.select([d.dan>=18,d.dan<=16],[1,0],default=np.nan)
common=pd.read_csv(common_path)
require('game_id' in common,'共通対局表の列不足')
forbidden=set(common.game_id)
raw23=add_cross(d[BASE_FEATURES])
eligible=(d.hour>cutoff)&d.target_label.notna()&~d.game_id.isin(forbidden)
quality=[]
for feature in FEATURES:
    x=raw23[feature]
    lower,upper=(-1,1) if feature=='chasing_riichi_quality_diff' else (0,14 if feature=='riichi_dora_mean' else 1)
    finite=x[np.isfinite(x)]
    quality.append(dict(feature=feature,scope='all_cached_games_before_imputation',n=len(x),missing=int(x.isna().sum()),
        missing_rate=x.isna().mean(),infinite=int(np.isinf(x).sum()),
        nonnumeric=int(bad_text[feature].sum()) if feature in BASE_FEATURES else np.nan,
        finite_unique=finite.nunique(),minimum=finite.min(),maximum=finite.max(),
        zero_rate_among_finite=finite.eq(0).mean(),expected_lower=lower,expected_upper=upper,
        out_of_range=int(((finite<lower-1e-10)|(finite>upper+1e-10)).sum())))
quality=CATALOG.merge(pd.DataFrame(quality),on='feature',validate='one_to_one')
quality.to_csv(out/'feature_quality.csv',index=False,encoding='utf-8-sig')
display(quality[['feature','区分','missing_rate','infinite','out_of_range','finite_unique']])
relations=pd.DataFrame([
    dict(check='流局2指標の和=1',eligible=int(raw23[['ryuukyoku_tenpai_rate','ryuukyoku_noten_safe_rate']].notna().all(axis=1).sum()),
         violations=int((raw23.ryuukyoku_tenpai_rate+raw23.ryuukyoku_noten_safe_rate-1).abs().gt(1e-10).sum())),
    dict(check='初打現物率+非現物率<=1（打牌前終了等で1未満はあり得る）',
         eligible=int(raw23[['first_discard_safe_rate','first_discard_raw_unsafe_rate']].notna().all(axis=1).sum()),
         violations=int((raw23.first_discard_safe_rate+raw23.first_discard_raw_unsafe_rate).gt(1+1e-10).sum()))])
relations.to_csv(out/'algebraic_checks.csv',index=False,encoding='utf-8-sig')
require(not bad_text.any().any() and not np.isinf(raw23.to_numpy()).any(),
        f'非数値・無限大を検出。品質表を保存して停止: {out}')
means,fit_ids=hour_means(d,cutoff,forbidden)
primary=prepare_hour_block(d,means,cutoff,'hour_invariant',forbidden)
saved=pd.read_csv(windows_path)
require({'game_id','source_ids','target_label'}<=set(saved.columns) and saved.game_id.is_unique,'窓一覧が不正')
require(set(primary['meta'].game_id)==set(saved.game_id),'再現した主方式の対象が既存H4と一致しません')
source_check=saved.set_index('game_id')
for r in primary['meta'].itertuples():
    prior=source_check.loc[r.game_id]
    require(set(ast.literal_eval(prior.source_ids))==set(r.source_ids) and int(prior.target_label)==r.target_label,
            '既存H4と元対局集合またはラベルが不一致')
means.to_csv(out/'shobon_only_imputation_means.csv')
primary['meta'].to_csv(out/'primary_windows.csv',index=False)
primary['excluded'].to_csv(out/'primary_exclusions.csv',index=False)
positions=d.reset_index().set_index('game_id').loc[primary['meta'].game_id,'index'].to_numpy(dtype=int)
used=set();nonoverlap=[]
for i,r in primary['meta'].iterrows():
    if not (set(r.source_ids)&used):nonoverlap.append(i);used.update(r.source_ids)
views={
    'primary_hour_invariant':(primary['X'],primary['y']),
    'raw_same_endpoints':(raw23.iloc[positions].reset_index(drop=True),primary['y']),
    'raw_all_eligible':(raw23.loc[eligible].reset_index(drop=True),d.loc[eligible,'target_label'].astype(int).reset_index(drop=True)),
    'primary_disjoint_windows':(primary['X'].iloc[nonoverlap].reset_index(drop=True),primary['y'].iloc[nonoverlap].reset_index(drop=True)),
}
primary['meta'].iloc[nonoverlap].to_csv(out/'disjoint_windows.csv',index=False)
reuse=Counter(g for seq in primary['meta'].source_ids for g in seq)
pd.DataFrame([dict(game_id=g,window_reuse=n) for g,n in reuse.items()]).to_csv(out/'source_reuse.csv',index=False)
dan_lookup=d.set_index('game_id').dan
mix=primary['meta'][['game_id','hour','target_label']].copy()
mix['history_dan17_count']=[sum(dan_lookup.loc[list(seq)].eq(17)) for seq in primary['meta'].source_ids]
mix['history_low_high_mixed']=[bool(dan_lookup.loc[list(seq)].le(16).any() and dan_lookup.loc[list(seq)].ge(18).any()) for seq in primary['meta'].source_ids]
mix.to_csv(out/'window_label_mixture.csv',index=False)
pd.crosstab(primary['meta'].hour.dt.year,primary['y']).to_csv(out/'primary_class_counts_by_year.csv')
metadata=dict(source_commit='c6e149986a3ac63ccc12a22dff03e0c4084bf7c5',report=str(report),
    shobon_cache=str(cache),input_sha256={p.name:digest(p) for p in paths},cutoff_hour=str(cutoff),
    window=10,label='dan<=16:0; dan>=18:1; dan17 excluded from endpoint, retained in history',
    order_verified_original=str(sfacts.iloc[0].order_verified_original),
    full_history_confirmed_original=str(sfacts.iloc[0].full_history_confirmed_original),
    primary_n=len(primary['X']),disjoint_n=len(nonoverlap),source_unique=len(reuse),max_source_reuse=max(reuse.values(),default=0),
    warnings='生涯履歴・同時間帯順序は未確認。初期1500は観測対局に基づく。統計的推測は行わない。')
(out/'analysis_provenance.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
print('主方式の対象・元対局集合がH4と一致。出力先:',out)

## E：クラス別の記述統計と分布

差は「高段位期−低段位期」。標準化差は両群の不偏分散を有効件数でプールした標準偏差で除算します。
各群2件未満またはプール分散0なら標準化差は欠損。これは検定統計量ではなく、p値・信頼区間は計算しません。
補完前データの平均は利用可能な事象のある試合に条件づけられ、欠損の多い列では選択の影響があります。
raw_same_endpointsは対局単位、主方式は履歴平均なので、同じ母数の推定ではありません。

In [ ]:
stats=[];differences=[]
for view,(x,y) in views.items():
    for feature in FEATURES:
        group_stats={}
        for label in [0,1]:
            values=x.loc[y.eq(label),feature]
            valid=values[np.isfinite(values)]
            r=dict(view=view,feature=feature,label=label,n=len(values),valid_n=len(valid),
                   mean=valid.mean(),median=valid.median(),std=valid.std(ddof=1),
                   missing_rate=values.isna().mean(),infinite=int(np.isinf(values).sum()))
            stats.append(r);group_stats[label]=r
        low,high=group_stats[0],group_stats[1]
        difference=high['mean']-low['mean'];pooled=np.nan
        if low['valid_n']>=2 and high['valid_n']>=2:
            pooled=np.sqrt(((low['valid_n']-1)*low['std']**2+(high['valid_n']-1)*high['std']**2)/
                           (low['valid_n']+high['valid_n']-2))
        smd=difference/pooled if np.isfinite(pooled) and pooled>0 else np.nan
        differences.append(dict(view=view,feature=feature,mean_high_minus_low=difference,standardized_difference=smd,
            missing_high_minus_low=high['missing_rate']-low['missing_rate'],pooled_std=pooled))
stats=pd.DataFrame(stats);differences=pd.DataFrame(differences)
stats.to_csv(out/'comparison_group_statistics.csv',index=False)
differences.to_csv(out/'comparison_differences.csv',index=False)
main= differences[differences.view.eq('primary_hour_invariant')].copy()
main['absolute_standardized_difference']=main.standardized_difference.abs()
main=main.sort_values('absolute_standardized_difference',ascending=False,na_position='last').merge(
    CATALOG[['feature','意味','解釈','定義上の懸念','区分']],on='feature',validate='one_to_one')
for v in ['raw_same_endpoints','raw_all_eligible','primary_disjoint_windows']:
    other=differences[differences.view.eq(v)].set_index('feature')
    main[v+'_smd']=main.feature.map(other.standardized_difference)
    main[v+'_same_direction']=np.where(main.standardized_difference.notna() & main[v+'_smd'].notna(),
        np.sign(main.standardized_difference).eq(np.sign(main[v+'_smd'])).astype(float),np.nan)
main.to_csv(out/'ranked_differences_and_robustness.csv',index=False,encoding='utf-8-sig')
display(stats[stats.view.eq('primary_hour_invariant')])
display(main.head(10))
# 日本語フォントを要求せず、グラフのラベルは既存変数名で統一。
for page in range(4):
    fig,axes=plt.subplots(3,2,figsize=(13,10))
    for ax,feature in zip(axes.flat,FEATURES[page*6:(page+1)*6]):
        for label,color,title in [(0,'tab:blue','Low rank'),(1,'tab:orange','High rank')]:
            values=np.sort(primary['X'].loc[primary['y'].eq(label),feature].dropna().to_numpy())
            if len(values):ax.step(values,np.arange(1,len(values)+1)/len(values),where='post',color=color,label=f'{title} (n={len(values)})')
        ax.set_title(feature,fontsize=9);ax.set_ylabel('Empirical cumulative proportion');ax.legend(fontsize=8)
    for ax in list(axes.flat)[len(FEATURES[page*6:(page+1)*6]):]:ax.set_visible(False)
    fig.suptitle('hour_invariant: descriptive distributions; overlapping windows')
    fig.tight_layout();fig.savefig(out/f'primary_distributions_{page+1}.png',dpi=140);plt.show();plt.close(fig)
fig,ax=plt.subplots(figsize=(11,7))
top=main.dropna(subset=['standardized_difference']).head(10).iloc[::-1]
ax.barh(top.feature,top.standardized_difference,color='tab:purple')
ax.axvline(0,color='black',linewidth=.8);ax.set_xlabel('Standardized difference: high minus low (descriptive)')
fig.tight_layout();fig.savefig(out/'top_standardized_differences.png',dpi=140);plt.show();plt.close(fig)

In [ ]:
lines=['# しょぼんｎの低段位期・高段位期の比較',
    '対象は同一人物の段位区分であり、一般的な強者・弱者全体の比較ではない。',
    f'主方式 {len(primary["X"])}窓、元対局非重複の補助分析 {len(nonoverlap)}窓。',
    f'主方式の元対局は{len(reuse)}件で、1対局の最大再利用回数は{max(reuse.values(),default=0)}。窓数は独立標本数ではない。',
    '## 記述的に差が大きい特徴量（|標準化差|順。統計的有意差や因果効果ではない）']
for r in main.dropna(subset=['standardized_difference']).head(10).itertuples():
    direction='高い' if r.mean_high_minus_low>0 else ('低い' if r.mean_high_minus_low<0 else '同じ')
    description=CATALOG.set_index('feature').loc[r.feature]
    lines.append(f'- {r.feature}（{description["意味"]}）：高段位期の平均が{direction}。平均差={r.mean_high_minus_low:.4g}、標準化差={r.standardized_difference:.3f}。'
                 f'打ち方の読み方：{description["解釈"]}の数値が変化した。注意：{description["定義上の懸念"]}')
lines += ['## 頑健性の確認',
    'ranked_differences_and_robustness.csvに補完前の同じ対象対局・全対象対局・元対局非重複窓との符号比較を保存した。',
    '符号が一致しても頑健性の証明ではない。補完、集約、対象選択、件数の違いを併せて確認する。',
    '主方式の欠損率が0でも元事象が全試合に存在する意味ではない。補完前の欠損率はcomparison_group_statistics.csvを参照。',
    '## 品質点検と提案',
    '23列は既存結果の再現用として維持できるが、名称通りの戦術を測っているとは限らず、そのまま実力差の説明に使わない。',
    '優先して修正を検討する対象：門前未リーチをノーテンとする判定、牌種数による良形判定、非現物と無筋の混同、Nの復号・手牌更新、宣言打牌前のドラ計数、流局非開示率を安全率と呼ぶ解釈。',
    '追加検討は分母の機会数と非発生の記録などの診断情報。今回は列・式・モデルを変更していない。',
    '## 考察の限界',
    '差の原因は段位と同時に変わった年代、卓・対戦相手、ルール、本人の状態等かもしれず、実力向上の因果効果とは断定できない。',
    '同時間帯の順序と生涯全履歴の完全性は未確認。観測初期1500対局の時間帯までを除外し、順序不変窓だけを用いた。',
    '段位17を履歴に含め、当該対局の段位で窓にラベルを付けた。窓内の段位混在はwindow_label_mixture.csvに保存。',
    '元対局非重複窓にも時期の相関が残る。p値、信頼区間、有意差、因果推定は行っていない。',
    '対局ごとの特徴量CSVには分母・分子の回数が残っていない。欠損原因や割合推定の精度はこのCSVだけでは検証できない。',
    '外部プレイヤーの性能は読込・選択基準に利用していない。',
    '## 次の分析',
    '主要な差を補完前の欠損率・分布と照合し、少数の既知局面で指標が説明通り動くか点検する。次に年代・卓条件をそろえられるか検討する。']
text='\n\n'.join(lines)
(out/'analysis_ja.md').write_text(text,encoding='utf-8')
display(text)
proposals=CATALOG[['feature','意味','区分','優先度','提案']].copy()
extra=pd.DataFrame([
    dict(feature='未実装：機会分母・分子の件数',意味='指標の推定精度を点検する補助記録',区分='追加検討',優先度=2,提案='将来の少量検証時に保存する。モデル入力への追加は今回行わない'),
    dict(feature='未実装：非発生・欠損理由',意味='行動なしと低い割合を区別する補助記録',区分='追加検討',優先度=2,提案='欠損理由を診断用に記録する案。今回の23列と補完式は維持'),
])
pd.concat([proposals,extra],ignore_index=True).to_csv(out/'feature_proposals.csv',index=False,encoding='utf-8-sig')
print('保存先:',out)
print('数値はこの実行で読み込んだ実データの結果です。既存H1〜H6・モデル・キャッシュは変更していません。')